# Hugging Face Hub Datasets

## Notebook Contract
- **Objective:** demonstrate the v0.3 Hub-dataset workflow: preset registry, split mapping, label-name normalization, and a baseline trained on real Hub data.
- **Inputs:** an in-memory mock `DatasetDict` for the offline path; the AG News dataset (or any other preset) for the opt-in real path.
- **Outputs:** dataset preview, baseline metrics, comparison summary written under `reports/sample_run/`.
- **Default mode:** offline. The opt-in flag `RUN_HUB_DOWNLOAD=True` triggers a real Hub download.
- **Expected runtime:** under 30 seconds offline; a few minutes on the real download path depending on dataset size.
- **Scope boundary:** all loading and normalization lives in `src/hf_finetuning_lab/data/hub.py`; the notebook orchestrates and visualizes.

## 1) Setup and Reproducibility

In [1]:
from __future__ import annotations

import os
import platform
import random
import sys
from datetime import UTC, datetime
from pathlib import Path

ROOT = Path('..').resolve()
SRC_PATH = ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.pipeline import Pipeline

from hf_finetuning_lab.data.hub import (
    HUB_PRESETS,
    HubDatasetConfig,
    list_hub_presets,
    load_hub_dataset,
    normalize_hub_dataset_dict,
)
from hf_finetuning_lab.data.io import validate_text_classification_frame

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

print(f'Python: {sys.version.split()[0]}')
print(f'Platform: {platform.platform()}')
print(f"Timestamp (UTC): {datetime.now(UTC).isoformat(timespec='seconds')}")

Python: 3.12.11
Platform: Windows-11-10.0.26200-SP0
Timestamp (UTC): 2026-05-17T20:30:09+00:00


## 2) Parameters

`RUN_HUB_DOWNLOAD=False` is the default so this notebook always executes in CI without network. Flip it to `True` to download the real dataset selected by `PRESET_NAME`.

In [2]:
REPORTS_DIR = ROOT / 'reports' / 'sample_run'
HUB_SUMMARY_PATH = REPORTS_DIR / 'hub_dataset_summary.csv'
HUB_METRICS_PATH = REPORTS_DIR / 'hub_baseline_metrics.json'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

PRESET_NAME = 'ag_news'
MAX_ROWS_PER_SPLIT = 1500
RUN_HUB_DOWNLOAD = False

print(f'Reports dir: {REPORTS_DIR}')
print(f'Preset: {PRESET_NAME!r}')
print(f'Max rows per split: {MAX_ROWS_PER_SPLIT}')
print(f'Run Hub download: {RUN_HUB_DOWNLOAD}')

Reports dir: C:\Users\diogo\work_code\portfolio\huggingface-finetuning-lab\reports\sample_run
Preset: 'ag_news'
Max rows per split: 1500
Run Hub download: False


## 3) Preset Registry

Each preset records the dataset name on the Hub, the source field for `text` / `label`, an optional config (for datasets with sub-tasks like `tweet_eval`), the split mapping, and the human-readable label names.

In [3]:
preset_rows = []
for name in list_hub_presets():
    cfg = HUB_PRESETS[name]
    preset_rows.append(
        {
            'preset': name,
            'hub_name': cfg.name,
            'config': cfg.config or '',
            'splits': ', '.join(f'{k}->{v}' for k, v in cfg.splits().items()),
            'n_labels': len(cfg.label_names) if cfg.label_names else None,
            'description': cfg.description,
        }
    )
preset_table = pd.DataFrame(preset_rows)
preset_table

,preset,hub_name,config,splits,n_labels,description
0,ag_news,ag_news,,"train->train, test->test",4.0,AG News topic classification (4 classes).
1,banking77,banking77,,"train->train, test->test",NaN,Banking customer-intent classification (77 fin...
2,imdb,imdb,,"train->train, test->test",2.0,IMDb sentiment classification (2 classes).
3,tweet_eval_sentiment,tweet_eval,sentiment,"train->train, test->test, validation->validation",3.0,TweetEval sentiment subtask (3 classes).


## 4) Offline Path: Normalize an In-Memory Mock

We build a tiny `DatasetDict` with the same shape AG News exposes on the Hub, then run it through the same normalization function used by `load_hub_dataset`. This exercises the API in CI without any network calls.

In [4]:
from datasets import ClassLabel, Dataset, DatasetDict, Features, Value

mock_train = pd.DataFrame(
    {
        'text': [
            'global summit ends in compromise',
            'team wins championship in overtime',
            'stock market closes at a record high',
            'researchers unveil new fusion technique',
            'trade talks resume after long pause',
            'underdog squad upsets the leaders',
            'central bank holds interest rates steady',
            'engineers test ultra-light battery',
        ],
        'label': [0, 1, 2, 3, 0, 1, 2, 3],
    }
)
mock_test = pd.DataFrame(
    {
        'text': [
            'diplomats meet to defuse border tension',
            'rookie pitcher delivers a no-hitter',
            'quarterly earnings beat estimates',
            'astronomers find unusual exoplanet',
        ],
        'label': [0, 1, 2, 3],
    }
)

ag_news_features = Features({'text': Value('string'), 'label': ClassLabel(names=list(HUB_PRESETS['ag_news'].label_names))})
mock_dict = DatasetDict(
    {
        'train': Dataset.from_pandas(mock_train, features=ag_news_features, preserve_index=False),
        'test': Dataset.from_pandas(mock_test, features=ag_news_features, preserve_index=False),
    }
)

offline_frames = normalize_hub_dataset_dict(mock_dict, HUB_PRESETS['ag_news'])
for split_name, frame in offline_frames.items():
    print(f"{split_name}: {len(frame)} rows | labels: {sorted(frame['label'].unique().tolist())}")
offline_frames['train'].head()

train: 8 rows | labels: ['Business', 'Sci/Tech', 'Sports', 'World']
test: 4 rows | labels: ['Business', 'Sci/Tech', 'Sports', 'World']


,text,label,label_id
0,global summit ends in compromise,World,0
1,team wins championship in overtime,Sports,1
2,stock market closes at a record high,Business,2
3,researchers unveil new fusion technique,Sci/Tech,3
4,trade talks resume after long pause,World,0


## 5) Load the Working Splits

Pick the real dataset if `RUN_HUB_DOWNLOAD=True`, otherwise stay on the mock so smoke runs offline. Either path produces the same canonical `text` / `label` / `label_id` schema.

In [5]:
if RUN_HUB_DOWNLOAD:
    frames = load_hub_dataset(PRESET_NAME, max_rows_per_split=MAX_ROWS_PER_SPLIT)
    source = f'hub:{PRESET_NAME}'
else:
    frames = offline_frames
    source = 'mock:ag_news (offline)'

train_df = frames['train']
test_df = frames['test']
validate_text_classification_frame(train_df, text_col='text', label_col='label')
validate_text_classification_frame(test_df, text_col='text', label_col='label')
print(f'Source:     {source}')
print(f'Train rows: {len(train_df)}')
print(f'Test rows:  {len(test_df)}')
print(f"Labels:     {sorted(train_df['label'].unique().tolist())}")
train_df.head()

Source:     mock:ag_news (offline)
Train rows: 8
Test rows:  4
Labels:     ['Business', 'Sci/Tech', 'Sports', 'World']


,text,label,label_id
0,global summit ends in compromise,World,0
1,team wins championship in overtime,Sports,1
2,stock market closes at a record high,Business,2
3,researchers unveil new fusion technique,Sci/Tech,3
4,trade talks resume after long pause,World,0


## 6) Quick EDA

In [6]:
summary_rows = []
for split_name, frame in frames.items():
    text_len = frame['text'].str.len()
    label_share = frame['label'].value_counts(normalize=True).round(4).to_dict()
    summary_rows.append(
        {
            'split': split_name,
            'rows': len(frame),
            'unique_labels': frame['label'].nunique(),
            'text_len_mean': float(text_len.mean()),
            'text_len_p95': float(text_len.quantile(0.95)),
            'label_share': label_share,
        }
    )
summary = pd.DataFrame(summary_rows)
summary.to_csv(HUB_SUMMARY_PATH, index=False)
print(f'Summary written to: {HUB_SUMMARY_PATH}')
summary

Summary written to: C:\Users\diogo\work_code\portfolio\huggingface-finetuning-lab\reports\sample_run\hub_dataset_summary.csv


,split,rows,unique_labels,text_len_mean,text_len_p95,label_share
0,train,8,4,35.375,39.65,"{'World': 0.25, 'Sports': 0.25, 'Business': 0...."
1,test,4,4,35.250,38.40,"{'World': 0.25, 'Sports': 0.25, 'Business': 0...."


## 7) Baseline on the Hub Schema

Same TF-IDF + LogReg baseline as the rest of the lab. With only 8 train rows in the offline mock the F1 numbers are illustrative; on a real preset they become meaningful.

In [7]:
baseline = Pipeline(
    steps=[
        ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=20000)),
        ('clf', LogisticRegression(max_iter=1000, random_state=SEED)),
    ]
)
baseline.fit(train_df['text'], train_df['label'])
pred = baseline.predict(test_df['text'])
macro_f1 = float(f1_score(test_df['label'], pred, average='macro', zero_division=0))
weighted_f1 = float(f1_score(test_df['label'], pred, average='weighted', zero_division=0))

metrics = {
    'source': source,
    'train_rows': int(len(train_df)),
    'test_rows': int(len(test_df)),
    'macro_f1': macro_f1,
    'weighted_f1': weighted_f1,
}
import json
HUB_METRICS_PATH.write_text(json.dumps(metrics, indent=2), encoding='utf-8')
print(f'Metrics written to: {HUB_METRICS_PATH}')
metrics

Metrics written to: C:\Users\diogo\work_code\portfolio\huggingface-finetuning-lab\reports\sample_run\hub_baseline_metrics.json


{'source': 'mock:ag_news (offline)',
 'train_rows': 8,
 'test_rows': 4,
 'macro_f1': 0.1,
 'weighted_f1': 0.1}

## 8) CLI Equivalents (opt-in)

The same loading path is exposed through the CLI for non-notebook workflows. These command strings are printed only; flip `RUN_HUB_DOWNLOAD` to actually execute them.

In [8]:
list_cmd = ['poetry', 'run', 'hf-lab', 'list-hub-datasets']
fetch_cmd = [
    'poetry', 'run', 'hf-lab', 'fetch-hub-dataset',
    '--name', PRESET_NAME,
    '--output-dir', str(ROOT / 'data' / 'hub' / PRESET_NAME),
    '--max-rows-per-split', str(MAX_ROWS_PER_SPLIT),
]
print('List presets:')
print(' '.join(list_cmd))
print('Fetch dataset to CSV:')
print(' '.join(fetch_cmd))

List presets:
poetry run hf-lab list-hub-datasets
Fetch dataset to CSV:
poetry run hf-lab fetch-hub-dataset --name ag_news --output-dir C:\Users\diogo\work_code\portfolio\huggingface-finetuning-lab\data\hub\ag_news --max-rows-per-split 1500


## 9) Checklist
- Preset registry exposes 4 datasets (AG News, IMDb, Banking77, TweetEval sentiment) with split mapping and label-name normalization.
- Offline path uses an in-memory `DatasetDict` mock so this notebook stays runnable in CI without network access.
- Opt-in real path (`RUN_HUB_DOWNLOAD=True`) downloads the selected preset and applies the same normalization.
- CLI exposes `list-hub-datasets` and `fetch-hub-dataset` for non-notebook use.
- For production: pin dataset revisions, document licensing, and rerun the v0.4 robust-evaluation notebook on the loaded splits.